# SPAM MAIL DETECTION

In [338]:
import pandas as pd
import re
import nltk
import pickle as pk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [339]:
mail_data=pd.read_csv('spam.csv',encoding='latin-1')

In [340]:
mail_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [341]:
mail_data.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [342]:
mail_data.shape

(5572, 5)

In [343]:
for column in mail_data.columns:
    print(f"Column: {column}")
    print(len(mail_data[column].unique()))


Column: v1
2
Column: v2
5169
Column: Unnamed: 2
44
Column: Unnamed: 3
11
Column: Unnamed: 4
6


In [344]:
mail_data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)

mail_data.rename(columns={'v1': 'label', 'v2': 'message'}, inplace=True)

In [345]:
mail_data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Label Encoding

In [346]:
mail_data['label'] = mail_data['label'].map({'spam': 1, 'ham': 0})

In [347]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [348]:
ps = PorterStemmer()

In [349]:
stop_words = set(stopwords.words('english'))

In [350]:
print(stopwords.words('english')[:10])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


In [351]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'http\S+', ' ', text)   
    text = re.sub(r'[^a-zA-Z]', ' ', text)   
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]    
    words = [ps.stem(word) for word in words]    
    return " ".join(words)

In [352]:
mail_data['processed_text'] = mail_data['message'].apply(preprocess_text)

AttributeError: 'int' object has no attribute 'lower'

spam - 0<br>
ham - 1

In [ ]:
X = mail_data['processed_text']
Y = mail_data['message']

In [ ]:
print(X)

In [ ]:
print(Y)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

In [ ]:
print(X.shape)
print(X_train.shape)
print(X_test.shape)

In [ ]:
vectorizer = TfidfVectorizer()
X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)


In [ ]:
print(X_train)

In [ ]:
print(X_train_features)

In [ ]:
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

In [ ]:
smote = SMOTE(random_state=42)

In [ ]:
X_train_smote, Y_train_smote = smote.fit_resample(X_train_features, Y_train)

In [ ]:
print(Y_train_smote.value_counts())

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(kernel='linear', probability=True),
}

In [ ]:
for name, model in models.items():

    print("-"*60)
    print(name)
    print("-"*60)

    model.fit(X_train_smote, Y_train_smote)

    train_pred = model.predict(X_train_features)

    print("\nTRAINING RESULTS")
    print("Accuracy:", accuracy_score(Y_train, train_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_train, train_pred))
    print("Classification Report:\n", classification_report(Y_train, train_pred))

    test_pred = model.predict(X_test_features)

    print("\nTEST RESULTS")
    print("Accuracy:", accuracy_score(Y_test, test_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, test_pred))
    print("Classification Report:\n", classification_report(Y_test, test_pred))

    print("\n\n")

In [ ]:
for name, model in models.items():

    y_prob = model.predict_proba(X_test_features)[:,1]

    auc_score = roc_auc_score(Y_test, y_prob)

    print(f"{name} ROC-AUC Score: {auc_score:.4f}")

In [ ]:
with open("spam_models.pkl", "wb") as file:
    pk.dump({
        "logistic_model": models["Logistic Regression"],
        "svm_model": models["SVM"],
        "vectorizer": vectorizer
    }, file)

In [ ]:
with open("spam_models.pkl", "rb") as file:
    data = pk.load(file)

logistic_model = data["logistic_model"]
svm_model = data["svm_model"]
vectorizer = data["vectorizer"]

In [ ]:
input_mail = ["Congratulations! You have won a FREE lottery worth $10000. Claim your prize now by clicking this link."]

input_features = vectorizer.transform(input_mail)

lr_pred = logistic_model.predict(input_features)
svm_pred = svm_model.predict(input_features)

print("Logistic Regression:", "Spam" if lr_pred[0] == 0 else "Ham")
print("SVM:", "Spam" if svm_pred[0] == 0 else "Ham")

In [ ]:
input_mail = ["Hi Shivangi, the meeting has been scheduled for tomorrow at 10 AM. Please join on time."]

input_features = vectorizer.transform(input_mail)

lr_pred = logistic_model.predict(input_features)
svm_pred = svm_model.predict(input_features)

print("Logistic Regression:", "Spam" if lr_pred[0] == 0 else "Ham")
print("SVM:", "Spam" if svm_pred[0] == 0 else "Ham")